# Quand l'équation reparle · *When the equation speaks again*

Notebook compagnon du chapitre **26. La théorie quantitative de la monnaie (MV = PQ) : relier monnaie, prix et activité** — [lire l'article](https://nmlab.io/ressources/theorie-quantitative-monnaie-mv-pq).
Companion notebook to chapter **26. The Quantity Theory of Money (MV = PQ): Linking Money, Prices and Activity** — [read the article](https://nmlab.io/en/ressources/quantity-theory-of-money).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    m2=load("M2SL","2014-06"); m2g=100*(m2/m2.shift(12)-1).dropna()
    cpi=load("CPIAUCSL","2014-06"); cpig=100*(cpi/cpi.shift(12)-1).dropna()
    fig=nm.figure(1010); ax=nm.axes(fig)
    ax.axhline(0,color=C["edge"],lw=1.4)
    ax.plot(m2g.index,m2g.values,color=C["blue"],lw=3,label=("Croissance de M2" if lang=="fr" else "M2 growth"))
    ax.plot(cpig.index,cpig.values,color=C["rose"],lw=3,label=("Inflation (IPC)" if lang=="fr" else "Inflation (CPI)"))
    d=dict(fr=("Quand la monnaie large reprend la parole","États-Unis, croissance sur un an de M2 et de l'IPC, en %.",
               "M2 : +26,8 %\n(fév. 2021)","inflation : 9,1 %\n(juin 2022)","Seize mois séparent les deux pics. Le lien monnaie-prix est fort quand l'inflation s'emballe,\nténu quand elle dort (BRI, 2023). Source : FRED (M2SL, CPIAUCSL)."),
           en=("When broad money speaks again","United States, year-on-year growth of M2 and CPI, in %.",
               "M2: +26.8%\n(Feb. 2021)","inflation: 9.1%\n(June 2022)","Sixteen months separate the two peaks. The money-price link is strong when inflation flares,\nfaint when it sleeps (BIS, 2023). Source: FRED (M2SL, CPIAUCSL)."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    xm=m2g.idxmax(); xi=cpig["2022":"2022"].idxmax()
    ax.annotate(t[2],xy=(xm,m2g.max()),xytext=(pd.Timestamp("2015-01-01"),20),fontsize=16,color=C["blue"],
                fontweight="bold",va="center",linespacing=1.2,arrowprops=dict(arrowstyle="-|>",color=C["blue"],lw=1.7))
    ax.annotate(t[3],xy=(xi,cpig[xi]),xytext=(pd.Timestamp("2023-09-01"),20),fontsize=16,color=C["rose"],
                fontweight="bold",va="center",linespacing=1.2,arrowprops=dict(arrowstyle="-|>",color=C["rose"],lw=1.7))
    leg=ax.legend(loc="upper right",fontsize=17,frameon=True,facecolor=C["bg"],edgecolor=C["edge"])
    for txt in leg.get_texts(): txt.set_color(C["text"])
    ax.set_ylim(-8,30)
    nm.footer(fig,t[4]);
    return fig


build_figure(LANG)